In [2]:
import numpy as np
from IPython.display import display, Markdown, Latex

In [41]:
a = np.uint8(127)

x = 0b11111111
np.int8(x)

OverflowError: Python integer 255 out of bounds for int8

In [236]:
e_x, e_y = 0, 0

m_x, m_y = 0.5, 0.5

x = 1 + m_x
y = 1 + m_y

print(1 + m_x + m_y + (m_x * m_y))
print(x*y)

2.25
2.25


In [239]:
bin(0x3F780000)

'0b111111011110000000000000000000'

In [241]:
bin(0x7FFFFFFF)

'0b1111111111111111111111111111111'

In [237]:
#L-mul

1 + m_x + m_y + 2**(-l(m))

2.0

$ float = -1^{sign} * 2^{exp} * (1 + mantissa) $

**`float8_e4m3`**
- sign: 1 bit
- exponent: 3 bits
- mantissa: 4 bits

Lets try calculating $1.5 * 2.5$ represented in `FP8` format by hand, step by step  

> Convert $1.5$ and $2.5$ to their floating point representation

Lets fill in the template `|_|_|_|_|_|_|_|_|` for each number  
$1.5$ -> $|0|001|1000|$

In [154]:
def float8_to_decimal(float8: str, e_bits: int = 4, exp_bias: int = 7, m_bits: int = 3):
    sign_bit = int(float8[0])
    sign = (-1)**sign_bit

    exponent_bits = float8[1:5]
    exponent = int(exponent_bits, 2)
    
    mantissa_bits = float8[5:]
    mantissa = int(mantissa_bits, 2)
    
    normal = 1
    if exponent == 0:
        exponent += 1
        normal = 0

    print('sign: +' if sign > 0 else 'sign: -')
    display(Latex(f"$ sign = -1^{{{sign_bit}}} = {{{(-1)**sign_bit}}} $"))
    display(Latex(f"$ exponent = 2^{{{exponent}-{exp_bias}}} = {{{2**(exponent-exp_bias)}}} $"))

    

    display(Latex(f"$ \\text{{mantissa}}: {{{mantissa_bits}}}_2 = {{{mantissa}}}_{{10}} \\rightarrow $"))

    print(f"exponent bits: {float8[1:5]}, biased: {exponent}, unbiased: {exponent-exp_bias}, (2^{exponent-exp_bias})={2**(exponent-exp_bias)}")
    print(f"mantissa bits: {int(float8[5:])}, binary value: {(1 + 2**(-m_bits) * mantissa)}")
    
    # Calculate the decimal value
    exp_str = f"{exponent}-{exp_bias}"
    display(Latex(f"$ float = -1^{{{sign_bit}}} * 2^{{{exponent}-{exp_bias}}} * (1 + 2^{{{-m_bits}}} * {{{mantissa}}}) $"))
    decimal_value =  2**(exponent - exp_bias) * (normal + 2**(-m_bits) * mantissa)
    return decimal_value

float8_to_decimal(float8='00000111')

sign: +


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

exponent bits: 0000, biased: 1, unbiased: -6, (2^-6)=0.015625
mantissa bits: 111, binary value: 1.875


<IPython.core.display.Latex object>

0.013671875

In [178]:
def display_mantissa_steps(mantissa_str):
    # Display original mantissa bits
    display(Latex(f"Mantissa Bits: ${mantissa_str}$"))
    
    # Display binary point notation 
    display(Latex(f"$=1.{mantissa_str}_2$"))
    
    # Build fraction representation
    fractions = []
    for i, bit in enumerate(mantissa_str):
        fractions.append(f"\\frac{{{bit}}}{{2^{i+1}}}")
    fraction_str = "+".join([f"1"] + fractions)
    display(Latex(f"$={fraction_str}$"))
    
    # Calculate decimal values
    decimals = []
    decimals.append("1")
    for i, bit in enumerate(mantissa_str):
        if bit == "1":
            val = round(1/2**(i+1), 8)
            decimals.append(str(val))
    decimal_str = "+".join(decimals)
    display(Latex(f"$={decimal_str}$"))
    
    # Display final result
    result = 1.0
    for i, bit in enumerate(mantissa_str):
        if bit == "1":
            result += 1/2**(i+1)
    display(Latex(f"$=\\fbox{{{result}}}$"))

display_mantissa_steps('101')

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

In [4]:
# if input is '110'
display(Latex("Mantissa Bits: $111$"))
display(Latex("$=1.110_2$"))
display(Latex("$=1+\\frac{1}{2^1}+\\frac{1}{2^2}+\\frac{0}{2^3}$"))
display(Latex("$=1+0.5+0.25$"))
display(Latex("$=\\fbox{1.75}$"))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

## 8-bit Floating Point Specification

OFP8 representation consists of sign, exponent, and mantissa fields. In this specification we
use the term mantissa to refer to the trailing significand bits. Two encodings are defined - E4M3
and E5M2, where the name explicitly states the number of bits in the exponent (E) and mantissa
(M) fields. Encodings consist of:
- 1 sign bit: the most significant bit
- e-bit biased exponent: 4 bits for E4M3, 5 bits for E5M2
- m mantissa (trailing significand) bits: 3 bits for E4M3, 2 bits for E5M2

The value, v, of a normal OFP8 number is
$$ v = (−1)^S × 2^{E−bias} × (1 + 2^{−m} × M) $$
The value, v, of a subnormal OFP8 number (subnormals have E = 0 and M > 0) is
$$ v = (−1)^S × 2^{1−bias} × (0 + 2^{−m} × M) $$

Exponent parameters and min/max values for both OFP8 formats are specified in Table 1.
The E5M2 format represents infinities and NaNs. Interpretation of the three mantissa values for
NaNs is not defined. The E4M3 format does not represent infinities and uses only two bit
patterns for NaN (a single mantissa-exponent bit pattern but allowing both values of the sign bit)
in order to increase emax to 8 and thus to increase the dynamic range by one binade. Various
values for OFP8 formats are detailed in Table 2.

### Table 1: OFP8 exponent parameters

| Parameter | E4M3 | E5M2 |
|-----------|------|------|
| Exponent bias | 7 | 15 |
| emax (unbiased) | 8 | 15 |
| emin (unbiased) | -6 | -14 |

### Table 2: OFP8 value encoding details

| Parameter | E4M3 | E5M2 |
|-----------|------|------|
| Infinities | N/A | S.11111.00₂ |
| NaN | S.1111.111₂ | S.11111.{01, 10, 11}₂ |
| Zeros | S.0000.000₂ | S.00000.00₂ |
| Max normal number | S.1111.110₂ = ±448 | S.11110.11₂ = ±57,344 |
| Min normal number | S.0001.000₂ = ±2⁻⁶ | S.00001.00₂ = ±2⁻¹⁴ |
| Max subnormal number | S.0000.111₂ = ±0.875 * 2⁻⁶ | S.00000.11₂ = ±0.75 * 2⁻¹⁴ |
| Min subnormal number | S.0000.001₂ = ±2⁻⁹ | S.00000.01₂ = ±2⁻¹⁶ |
| Dynamic range | 18 binades | 32 binades |

In [201]:
def display_sign_steps(bits):
    sign_bit = bits[0]
    display(Latex(f"Sign Bit: ${sign_bit}$"))
    display(Latex(f"$(-1)^{sign_bit} = {'-' if sign_bit=='1' else '+'}1$"))

def display_exponent_steps(bits, format="E4M3"):
    if format == "E4M3":
        exp_bits = bits[1:5]
        bias = 7
    else: # E5M2
        exp_bits = bits[1:6] 
        bias = 15
        
    exp_val = int(exp_bits, 2)
    display(Latex(f"Exponent Bits: ${exp_bits}$"))
    display(Latex(f"$={exp_bits}_2 = {exp_val}_{{10}}$"))
    
    if exp_val == 0:
        # Subnormal case
        display(Latex(f"Subnormal case: $E = 0$ so use $2^{{1-bias}}$"))
        display(Latex(f"$2^{{1-{bias}}} = 2^{{{1-bias}}}$"))
    else:
        # Normal case
        display(Latex(f"$2^{{E-bias}} = 2^{{{exp_val}-{bias}}} = 2^{{{exp_val-bias}}}$"))

def display_mantissa_steps(bits, format="E4M3"):
    if format == "E4M3":
        mantissa_bits = bits[5:]
        m = 3
    else: # E5M2
        mantissa_bits = bits[6:]
        m = 2
        
    exp_val = int(bits[1:5], 2) if format == "E4M3" else int(bits[1:6], 2)
    
    display(Latex(f"Mantissa Bits: ${mantissa_bits}$"))
    
    M = int(mantissa_bits, 2)
    display(Latex(f"$M = {mantissa_bits}_2 = {M}_{{10}}$"))
    
    if exp_val == 0:
        # Subnormal case
        display(Latex(f"Subnormal case: $(0 + 2^{{-{m}}} × {M})$"))
        mantissa_val = (0 + 2**(-m) * M)
        display(Latex(f"$= {mantissa_val}$"))
    else:
        # Normal case
        display(Latex(f"$(1 + 2^{{-{m}}} × {M})$"))
        mantissa_val = (1 + 2**(-m) * M)
        display(Latex(f"$= {mantissa_val}$"))

def display_float8_conversion(bits, format="E4M3"):
    display(Latex(f"Converting {format} number: {bits}"))
    display(Latex("Step 1: Sign"))
    display_sign_steps(bits)
    
    display(Latex("Step 2: Exponent"))
    display_exponent_steps(bits, format)
    
    display(Latex("Step 3: Mantissa"))
    display_mantissa_steps(bits, format)
    
    # Calculate final value
    sign_bit = bits[0]
    sign = -1 if sign_bit == '1' else 1
    
    if format == "E4M3":
        exp_bits = bits[1:5]
        mantissa_bits = bits[5:]
        bias = 7
        m = 3
    else:
        exp_bits = bits[1:6]
        mantissa_bits = bits[6:]
        bias = 15
        m = 2
        
    exp_val = int(exp_bits, 2)
    M = int(mantissa_bits, 2)
    
    if exp_val == 0 and M > 0:
        # Subnormal
        value = sign * (2**(1-bias)) * (0 + 2**(-m) * M)
    elif M == 0:
        # Zero or negative zero
        value = 0
    else:
        # Normal
        value = sign * (2**(exp_val-bias)) * (1 + 2**(-m) * M)
        
    display(Latex(f"Final Value: $\\fbox{{{value}}}$"))


In [202]:
display_float8_conversion('00000000')

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

...

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

1 0.015625 0.0


<IPython.core.display.Latex object>

In [296]:
import struct
import numpy as np

def float_to_binary(num):
    # Using struct to get IEEE 754 binary representation
    binary = ''.join(f'{b:08b}' for b in struct.pack('!f', num))
    return binary

def float_to_binary_np(num):
    # Using numpy's binary_repr on float view
    return np.binary_repr(np.float16(num).view(np.int16), width=16)

# Example usage
num = 0b100000 # 2^(-1) = 0.5

# Method 1: struct
print(f"Using struct: {float_to_binary(num)}")

# Method 2: numpy 
print(f"Using numpy: {float_to_binary_np(num)}")

# For just the hex representation, you can use float.hex()
print(f"Hex representation: {num.hex()}")

Using struct: 01000010000000000000000000000000
Using numpy: 0101000000000000


AttributeError: 'int' object has no attribute 'hex'

In [328]:
def binary_to_float16(binary_str):
    # Remove any '0b' prefix and ensure string is 16 bits
    binary_str = binary_str.replace('0b', '').zfill(16)
    
    # Convert binary string to integer
    bits = int(binary_str, 2)
    
    # Create float16 from bits using numpy
    float_val = np.frombuffer(
        np.array([bits], dtype=np.uint16).tobytes(),
        dtype=np.float16
    )[0]
    
    # Display the conversion
    display(Latex(f"Binary input: ${binary_str}$"))
    display(Latex(f"Sign bit: ${binary_str[0]}$"))
    display(Latex(f"Exponent bits: ${binary_str[1:6]}$"))
    display(Latex(f"Mantissa bits: ${binary_str[6:]}$"))
    display(Latex(f"Value: $\\fbox{{{float_val}}}$"))
    
    return float_val

# Example usage
binary_to_float16("0011110000000000")  # This would represent 1.0
binary_to_float16("0011000000000000")  # This would represent 0.25
binary_to_float16('0101000000000000')


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

...

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

np.float16(32.0)

In [333]:
def binary_to_float32(binary_str):
    # Remove any '0b' prefix and ensure string is 16 bits
    binary_str = binary_str.replace('0b', '').zfill(32)
    
    # Convert binary string to integer
    bits = int(binary_str, 2)
    
    # Create float32 from bits using numpy
    float_val = np.frombuffer(
        np.array([bits], dtype=np.uint32).tobytes(),
        dtype=np.float32
    )[0]
    return float_val

In [339]:
binary_to_float32(float_to_binary(a))

np.float32(1.5)

In [343]:
a, b = 100.5, -287.445
mul = a * b

lmul = bin(int(float_to_binary(a),2) + int(float_to_binary(b),2)-0x3F780000)
lmul = binary_to_float32(lmul)

print(f"Multiplication: {mul}")
print(f"Logarithmic multiplication: {lmul}")
print(f"Difference: {abs(mul - lmul)}")

Multiplication: -28888.2225
Logarithmic multiplication: -28764.48046875
Difference: 123.7421875


In [362]:
offset32 = bin(0x3F780000)[2:]
offset32_exponent = offset32[:8]
offset32_mantissa = offset32[10:]
len(offset32)

30

In [ ]:
bin(0x3F780000)[10:10]

In [289]:
0b1011111100

764

In [252]:
binnp.float16(1.5).view(np.int16)

'0b11111000000000'

In [226]:
from decimal import Decimal

# 0.5 as binary 0.1
half = Decimal('0.5')

# 0.25 as binary 0.01
quarter = Decimal('0.25')
quarter.radix()

Decimal('10')

In [63]:
1 + 2**(-4) * 8

1.5

In [29]:
# Your binary string
binary_string = '000011'

# Convert binary string to a 32-bit integer
int_representation = int(binary_string, 2)

# Convert the integer to a 4-byte array (little-endian by default in numpy)
float_value = np.frombuffer(int_representation.to_bytes(4, byteorder='big'), dtype=np.float32)[0]

print(float_value)

3.761582e-37
